# Load-flow library experiments: PowerModels.jl vs PyPowSyBl/OpenLoadFlow

Purpose: run small, controlled experiments with official-style examples before integrating either library into the main repository.

This notebook is intentionally exploratory. It should answer:

- What does each library do by default?
- How do slack-bus choices affect results?
- How do reactive limits and PV/PQ switching behave?
- What must be configured explicitly for reliable comparison?
- What outputs are available for validation?

Official anchors checked while preparing this notebook:

- PowerModels power-flow docs: `solve_pf`, `solve_ac_pf`, `compute_ac_pf`, warm starts, and `calc_branch_flow_ac`.
- PyPowSyBl load-flow docs: `loadflow.run_ac`, `Parameters`, provider parameters, result objects, built-in example networks.
- OpenLoadFlow docs: AC calculations are based on full Newton-Raphson and support distributed slack, generator limits, voltage initialisation modes, and voltage/reactive controls.

## How to use this notebook

Run the Python/PyPowSyBl sections directly in a Python kernel.

For PowerModels.jl, this notebook writes Julia scripts into `experiments/powermodels_loadflow_examples/` and runs them with `julia` through `subprocess`. If Julia or the required Julia packages are not installed, the notebook will still create the scripts and print the command to run manually.

Expected local packages:

Python:
```bash
pip install pypowsybl pandas
```

Julia:
```julia
import Pkg
Pkg.add(["PowerModels", "Ipopt", "NLsolve", "JSON3", "DataFrames", "CSV"])
```

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import os
import textwrap
import shutil
import pandas as pd

ROOT = Path.cwd()
EXP = ROOT / "experiments"
PYPS = EXP / "pypowsybl_openloadflow_examples"
PM = EXP / "powermodels_loadflow_examples"
PYPS.mkdir(parents=True, exist_ok=True)
PM.mkdir(parents=True, exist_ok=True)

print("Experiment folders:")
print(PYPS)
print(PM)

# Part A — PyPowSyBl/OpenLoadFlow experiments

The official PyPowSyBl docs expose built-in network constructors such as `create_ieee14`, `create_ieee30`, and `create_eurostag_tutorial_example1_network`. The load-flow docs show that the default provider is OpenLoadFlow, and that `lf.Parameters()` exposes important defaults such as `distributed_slack=True` and `use_reactive_limits=True`.

We begin by inspecting what is available in the installed version.

In [ ]:
try:
    import pypowsybl as pp
    import pypowsybl.network as pn
    import pypowsybl.loadflow as lf
    print("pypowsybl version:", getattr(pp, "__version__", "unknown"))
    print("Load-flow providers:", lf.get_provider_names())
    print("Default provider:", lf.get_default_provider())
    print("Default Parameters:")
    print(lf.Parameters())
    print("\nProvider parameter names, first 40:")
    print(lf.get_provider_parameters_names()[:40])
except Exception as e:
    print("PyPowSyBl import/inspection failed:")
    print(repr(e))

## A1. Helper functions for PyPowSyBl result extraction

The exact network tables available can vary by version and by network. These helpers try to extract buses, generators, loads, and branch-like tables defensively.

In [ ]:
def safe_call(obj, method_name, *args, **kwargs):
    method = getattr(obj, method_name, None)
    if method is None:
        return None
    try:
        return method(*args, **kwargs)
    except Exception as exc:
        print(f"{method_name} failed: {exc!r}")
        return None

def component_results_to_dicts(results):
    rows = []
    for r in results:
        row = {
            "connected_component_num": getattr(r, "connected_component_num", None),
            "synchronous_component_num": getattr(r, "synchronous_component_num", None),
            "status": str(getattr(r, "status", None)),
            "status_text": getattr(r, "status_text", None),
            "iteration_count": getattr(r, "iteration_count", None),
            "reference_bus_id": getattr(r, "reference_bus_id", None),
            "distributed_active_power": getattr(r, "distributed_active_power", None),
        }
        slack_results = getattr(r, "slack_bus_results", []) or []
        row["slack_bus_ids"] = ",".join([getattr(s, "id", "") for s in slack_results])
        row["slack_active_power_mismatch"] = sum([
            float(getattr(s, "active_power_mismatch", 0.0) or 0.0) for s in slack_results
        ])
        rows.append(row)
    return rows

def extract_pypowsybl_tables(network):
    tables = {}
    for name in [
        "get_buses", "get_voltage_levels", "get_generators", "get_loads",
        "get_lines", "get_2_windings_transformers", "get_shunt_compensators",
        "get_vsc_converter_stations", "get_static_var_compensators"
    ]:
        df = safe_call(network, name)
        if df is not None:
            tables[name.replace("get_", "")] = df.copy()
    return tables

def save_tables(tables, folder, prefix):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    for name, df in tables.items():
        try:
            df.to_csv(folder / f"{prefix}_{name}.csv")
        except Exception as exc:
            print(f"Could not save {name}: {exc!r}")

## A2. Baseline PyPowSyBl example: official tutorial network

This mirrors the documentation style:

```python
network = pn.create_eurostag_tutorial_example1_network()
results = lf.run_ac(network, parameters=lf.Parameters(distributed_slack=False))
```

First run: controlled single-slack, reactive limits disabled/enabled comparison.

In [ ]:
def run_pypowsybl_case(case_name, network_factory, parameters, provider="OpenLoadFlow", report=False):
    import pypowsybl.report as report_mod
    network = network_factory()
    report_node = report_mod.ReportNode() if report else None
    results = lf.run_ac(network, parameters=parameters, provider=provider, report_node=report_node)
    summary = pd.DataFrame(component_results_to_dicts(results))
    tables = extract_pypowsybl_tables(network)
    out = PYPS / case_name
    out.mkdir(parents=True, exist_ok=True)
    summary.to_csv(out / "component_results.csv", index=False)
    save_tables(tables, out, "solved")
    if report_node is not None:
        (out / "report.txt").write_text(str(report_node), encoding="utf-8")
    return network, results, summary, tables, str(report_node) if report_node is not None else None

try:
    controlled_params = lf.Parameters(
        distributed_slack=False,
        use_reactive_limits=False,
        transformer_voltage_control_on=False,
        phase_shifter_regulation_on=False,
        shunt_compensator_voltage_control_on=False,
        provider_parameters={
            "acSolverType": "NEWTON_RAPHSON"
        }
    )
    net, res, summary, tables, rep = run_pypowsybl_case(
        "A2_eurostag_controlled_single_slack",
        pn.create_eurostag_tutorial_example1_network,
        controlled_params,
        report=True
    )
    display(summary)
    print(rep[:2000] if rep else "No report")
    display(tables.get("buses", pd.DataFrame()).head())
    display(tables.get("generators", pd.DataFrame()).head())
except Exception as e:
    print("PyPowSyBl baseline experiment failed:")
    print(repr(e))

## A3. PyPowSyBl: compare single slack vs distributed slack

Hypothesis: with distributed slack enabled, generator active powers may change because the active-power imbalance is distributed rather than absorbed by one slack source.

In [ ]:
try:
    experiments = {
        "single_slack": lf.Parameters(
            distributed_slack=False,
            use_reactive_limits=False,
            provider_parameters={"acSolverType": "NEWTON_RAPHSON"}
        ),
        "distributed_slack_default": lf.Parameters(
            distributed_slack=True,
            use_reactive_limits=False,
            provider_parameters={"acSolverType": "NEWTON_RAPHSON"}
        ),
    }

    gen_outputs = []
    comp_outputs = []
    for label, params in experiments.items():
        net, res, summary, tables, _ = run_pypowsybl_case(
            f"A3_eurostag_{label}",
            pn.create_eurostag_tutorial_example1_network,
            params
        )
        comp_outputs.append(summary.assign(experiment=label))
        gens = tables.get("generators", pd.DataFrame()).copy()
        if not gens.empty:
            gens["experiment"] = label
            gen_outputs.append(gens.reset_index())

    if comp_outputs:
        display(pd.concat(comp_outputs, ignore_index=True))
    if gen_outputs:
        gen_compare = pd.concat(gen_outputs, ignore_index=True)
        display(gen_compare)
        gen_compare.to_csv(PYPS / "A3_slack_generator_comparison.csv", index=False)
except Exception as e:
    print("Slack comparison failed:")
    print(repr(e))

## A4. PyPowSyBl: inspect provider parameters

Run this before depending on any provider parameter names. Different installed versions may expose different names/defaults.

In [ ]:
try:
    params_df = lf.get_provider_parameters()
    display(params_df)
    params_df.to_csv(PYPS / "openloadflow_provider_parameters.csv")
    interesting = [
        "acSolverType", "slackBusSelectionMode", "slackBusesIds",
        "reactiveLimitsMaxPqPvSwitch", "maxOuterLoopIterations",
        "voltageRemoteControl", "transformerVoltageControlMode"
    ]
    existing = [p for p in interesting if p in params_df.index]
    if existing:
        display(params_df.loc[existing])
except Exception as e:
    print("Provider parameter inspection failed:")
    print(repr(e))

## A5. PyPowSyBl: reactive limits and PV/PQ switching stress test

This section uses the Eurostag tutorial network variant with more generators / power limits when available. It compares `use_reactive_limits=False` and `True`.

The exact way to observe PV/PQ switching depends on PyPowSyBl/OpenLoadFlow version. The most robust first check is to compare generator `q` outputs, voltage magnitudes, and the OpenLoadFlow report.

In [ ]:
try:
    network_factories = {
        "eurostag_base": pn.create_eurostag_tutorial_example1_network,
    }
    if hasattr(pn, "create_eurostag_tutorial_example1_with_more_generators_network"):
        network_factories["eurostag_more_generators"] = pn.create_eurostag_tutorial_example1_with_more_generators_network
    if hasattr(pn, "create_eurostag_tutorial_example1_with_power_limits_network"):
        network_factories["eurostag_power_limits"] = pn.create_eurostag_tutorial_example1_with_power_limits_network

    rows = []
    for factory_name, factory in network_factories.items():
        for flag in [False, True]:
            label = f"A5_{factory_name}_reactive_limits_{flag}"
            params = lf.Parameters(
                distributed_slack=False,
                use_reactive_limits=flag,
                transformer_voltage_control_on=False,
                phase_shifter_regulation_on=False,
                shunt_compensator_voltage_control_on=False,
                provider_parameters={
                    "acSolverType": "NEWTON_RAPHSON",
                    # keep string values in provider_parameters
                    "reactiveLimitsMaxPqPvSwitch": "3",
                }
            )
            net, res, summary, tables, report_text = run_pypowsybl_case(label, factory, params, report=True)
            s = summary.copy()
            s["factory"] = factory_name
            s["use_reactive_limits"] = flag
            rows.append(s)
            print("\n", "="*80)
            print(label)
            print(report_text[:2500] if report_text else "No report")
            gens = tables.get("generators", pd.DataFrame())
            buses = tables.get("buses", pd.DataFrame())
            if not gens.empty:
                display(gens)
            if not buses.empty:
                display(buses)

    if rows:
        display(pd.concat(rows, ignore_index=True))
except Exception as e:
    print("Reactive-limit experiment failed:")
    print(repr(e))

## A6. PyPowSyBl: IEEE 14 stress test

Increase loads moderately and test convergence, voltages, and reactive-limit effects.

Note: PyPowSyBl network update APIs vary. This cell first prints the load table so you can adapt the modification method if needed.

In [ ]:
try:
    net = pn.create_ieee14()
    loads = net.get_loads()
    print("IEEE14 loads before modification:")
    display(loads.head())
    print("Available network methods containing 'update':")
    print([m for m in dir(net) if "update" in m.lower()][:50])
except Exception as e:
    print("Could not inspect IEEE14 loads/update API:")
    print(repr(e))

In [ ]:
# Version-dependent load-scaling template.
# If your installed PyPowSyBl exposes network.update_loads(df), uncomment/adapt this.

try:
    def ieee14_scaled_load_factory(scale_p=1.10, scale_q=1.10):
        net = pn.create_ieee14()
        loads = net.get_loads()
        modified = loads.copy()
        for col, scale in [("p0", scale_p), ("q0", scale_q)]:
            if col in modified.columns:
                modified[col] = modified[col] * scale
        # Try common update method names defensively.
        if hasattr(net, "update_loads"):
            net.update_loads(modified)
        else:
            raise RuntimeError("No update_loads method found in installed PyPowSyBl version. Inspect dir(net).")
        return net

    params = lf.Parameters(
        distributed_slack=False,
        use_reactive_limits=True,
        provider_parameters={"acSolverType": "NEWTON_RAPHSON"}
    )
    net, res, summary, tables, report_text = run_pypowsybl_case(
        "A6_ieee14_scaled_load_110pct",
        lambda: ieee14_scaled_load_factory(1.10, 1.10),
        params,
        report=True
    )
    display(summary)
    display(tables.get("buses", pd.DataFrame()).head())
except Exception as e:
    print("IEEE14 scaled-load experiment not executed:")
    print(repr(e))
    print("This is expected if your PyPowSyBl version uses a different update API.")

## A7. PyPowSyBl: load-flow validation API

The docs list `pypowsybl.loadflow.run_validation`. Use it to check whether the stored network values satisfy load-flow consistency checks after `run_ac`.

In [ ]:
try:
    net = pn.create_eurostag_tutorial_example1_network()
    params = lf.Parameters(distributed_slack=False, provider_parameters={"acSolverType": "NEWTON_RAPHSON"})
    results = lf.run_ac(net, parameters=params, provider="OpenLoadFlow")
    if hasattr(lf, "run_validation"):
        validation = lf.run_validation(net)
        print(validation)
    else:
        print("lf.run_validation is not available in this installed version.")
except Exception as e:
    print("Validation experiment failed:")
    print(repr(e))

# Part B — PowerModels.jl experiments

The PowerModels docs distinguish:

- `solve_ac_pf`: JuMP-based AC power flow, generally more robust.
- `compute_ac_pf`: native AC power flow using NLsolve, no JuMP model.
- `set_ac_pf_start_values!`: use current solution values as starts.
- `calc_branch_flow_ac`: required because branch flows are not produced by default.

This notebook writes Julia scripts to reproduce these experiments.

In [ ]:
powermodels_script = r'''
using PowerModels
using Ipopt
using NLsolve
using JSON3
using Dates

function safe_get(d, keys...)
    x = d
    for k in keys
        if haskey(x, k)
            x = x[k]
        else
            return missing
        end
    end
    return x
end

function summarize_result(label, result)
    println("\n", "="^80)
    println(label)
    println("status: ", get(result, "status", missing))
    println("termination_status: ", get(result, "termination_status", missing))
    println("solve_time: ", get(result, "solve_time", missing))
    println("objective: ", get(result, "objective", missing))
    if haskey(result, "solution")
        sol = result["solution"]
        println("solution keys: ", collect(keys(sol)))
        if haskey(sol, "bus")
            println("bus solution sample:")
            for (i, b) in first(collect(sol["bus"]), min(5, length(sol["bus"])))
                println(i, " => ", b)
            end
        end
        if haskey(sol, "gen")
            println("gen solution sample:")
            for (i, g) in first(collect(sol["gen"]), min(5, length(sol["gen"])))
                println(i, " => ", g)
            end
        end
    end
end

function write_json(path, obj)
    open(path, "w") do io
        JSON3.pretty(io, obj)
    end
end

outdir = joinpath(@__DIR__, "outputs")
mkpath(outdir)

case_file = joinpath(dirname(pathof(PowerModels)), "..", "test", "data", "matpower", "case5.m")
if !isfile(case_file)
    println("Could not find PowerModels test case5.m at: ", case_file)
    println("Set case_file manually to your local MATPOWER case.")
    exit()
end

println("Using case file: ", case_file)
data = PowerModels.parse_file(case_file)

println("\nPowerModels version check:")
try
    using Pkg
    deps = Pkg.dependencies()
    for (uuid, dep) in deps
        if dep.name in ["PowerModels", "Ipopt", "NLsolve"]
            println(dep.name, " => ", dep.version)
        end
    end
catch e
    println("Version inspection failed: ", e)
end

println("\nBus types:")
for (id, bus) in sort(collect(data["bus"]), by=x -> parse(Int, x[1]))
    println(id, " bus_type=", bus["bus_type"], " vm=", get(bus, "vm", missing), " va=", get(bus, "va", missing))
end

println("\nExperiment B1: compute_ac_pf default NLsolve method")
data_b1 = deepcopy(data)
r1 = PowerModels.compute_ac_pf(data_b1)
summarize_result("B1 compute_ac_pf default", r1)
write_json(joinpath(outdir, "B1_compute_ac_pf_default.json"), r1)

println("\nExperiment B2: compute_ac_pf method=:newton")
data_b2 = deepcopy(data)
r2 = PowerModels.compute_ac_pf(data_b2; method=:newton, show_trace=true, iterations=50, ftol=1e-8)
summarize_result("B2 compute_ac_pf method=:newton", r2)
write_json(joinpath(outdir, "B2_compute_ac_pf_newton.json"), r2)

println("\nExperiment B3: solve_ac_pf with Ipopt")
data_b3 = deepcopy(data)
r3 = PowerModels.solve_ac_pf(data_b3, Ipopt.Optimizer)
summarize_result("B3 solve_ac_pf Ipopt", r3)
write_json(joinpath(outdir, "B3_solve_ac_pf_ipopt.json"), r3)

println("\nExperiment B4: branch-flow calculation after compute_ac_pf")
data_b4 = deepcopy(data)
r4 = PowerModels.compute_ac_pf(data_b4; method=:newton)
PowerModels.update_data!(data_b4, r4["solution"])
flows = PowerModels.calc_branch_flow_ac(data_b4)
PowerModels.update_data!(data_b4, flows)
println("Calculated branch-flow keys sample:")
for (i, br) in first(collect(data_b4["branch"]), min(5, length(data_b4["branch"])))
    println(i, " => pf=", get(br, "pf", missing), " qf=", get(br, "qf", missing),
            " pt=", get(br, "pt", missing), " qt=", get(br, "qt", missing))
end
write_json(joinpath(outdir, "B4_network_with_branch_flows.json"), data_b4)

println("\nExperiment B5: warm-start from previous solution")
data_b5 = deepcopy(data)
r5a = PowerModels.compute_ac_pf(data_b5; method=:newton)
PowerModels.update_data!(data_b5, r5a["solution"])
PowerModels.set_ac_pf_start_values!(data_b5)
r5b = PowerModels.compute_ac_pf(data_b5; method=:newton)
summarize_result("B5 warm-start compute_ac_pf", r5b)
write_json(joinpath(outdir, "B5_warm_started.json"), r5b)

println("\nExperiment B6: tighten generator reactive limits and observe native behaviour")
data_b6 = deepcopy(data)
# Tighten qmax/qmin aggressively for the first non-reference generator if possible.
gen_ids = sort(collect(keys(data_b6["gen"])), by=x -> parse(Int, x))
if length(gen_ids) > 0
    gid = gen_ids[1]
    println("Tightening generator ", gid, " qmin/qmax around zero")
    data_b6["gen"][gid]["qmin"] = -0.01
    data_b6["gen"][gid]["qmax"] = 0.01
end
r6 = PowerModels.compute_ac_pf(data_b6; method=:newton)
summarize_result("B6 compute_ac_pf tightened Q limits", r6)
write_json(joinpath(outdir, "B6_tight_q_limits.json"), r6)
println("Inspect whether native compute_ac_pf enforced PV/PQ switching or only returned solved qg values.")

println("\nAll PowerModels experiments completed.")
'''
script_path = PM / "powermodels_loadflow_experiments.jl"
script_path.write_text(powermodels_script, encoding="utf-8")
print(f"Wrote Julia script to: {script_path}")
print("Run manually with:")
print(f"julia {script_path}")

In [ ]:
# Try running the Julia script automatically if Julia is available.
julia_exe = shutil.which("julia")
if julia_exe is None:
    print("Julia executable not found on PATH. Run the script manually in your Julia environment:")
    print(f"julia {PM / 'powermodels_loadflow_experiments.jl'}")
else:
    print("Running Julia script...")
    proc = subprocess.run(
        [julia_exe, str(PM / "powermodels_loadflow_experiments.jl")],
        capture_output=True,
        text=True,
        cwd=str(PM),
        timeout=180
    )
    print("STDOUT:")
    print(proc.stdout[-5000:])
    print("STDERR:")
    print(proc.stderr[-5000:])
    print("Return code:", proc.returncode)

# Part C — Comparison checklist

Fill this table after running the experiments.

In [ ]:
comparison_columns = [
    "feature",
    "PowerModels behaviour",
    "PyPowSyBl/OpenLoadFlow behaviour",
    "exact parameter/function",
    "default setting",
    "tested result",
    "implication for repository",
]
comparison = pd.DataFrame([
    {
        "feature": "Native Newton-Raphson AC PF",
        "PowerModels behaviour": "compute_ac_pf via NLsolve; set method=:newton for classical Newton",
        "PyPowSyBl/OpenLoadFlow behaviour": "OpenLoadFlow AC calculations use full Newton-Raphson by default/provider setting",
        "exact parameter/function": "PowerModels.compute_ac_pf(...; method=:newton) / lf.run_ac(... provider_parameters={'acSolverType':'NEWTON_RAPHSON'})",
        "default setting": "PowerModels NLsolve default may not be classical Newton; OpenLoadFlow default provider is OpenLoadFlow",
        "tested result": "",
        "implication for repository": "Always configure solver explicitly for controlled comparisons",
    },
    {
        "feature": "Branch flows",
        "PowerModels behaviour": "Not included by default; must call calc_branch_flow_ac after update_data!",
        "PyPowSyBl/OpenLoadFlow behaviour": "Network branch tables updated after run_ac",
        "exact parameter/function": "PowerModels.calc_branch_flow_ac / network.get_lines()",
        "default setting": "",
        "tested result": "",
        "implication for repository": "Normalize branch-flow extraction carefully",
    },
    {
        "feature": "Distributed slack",
        "PowerModels behaviour": "No comparable native distributed-slack outer loop in compute_ac_pf",
        "PyPowSyBl/OpenLoadFlow behaviour": "distributed_slack parameter; default True in docs",
        "exact parameter/function": "lf.Parameters(distributed_slack=False/True)",
        "default setting": "PyPowSyBl docs show distributed_slack=True",
        "tested result": "",
        "implication for repository": "Disable distributed slack for single-slack cross-validation",
    },
    {
        "feature": "Reactive limits / PV-PQ switching",
        "PowerModels behaviour": "No complete native PV/PQ switching outer loop in compute_ac_pf",
        "PyPowSyBl/OpenLoadFlow behaviour": "ReactiveLimits outer loop available via use_reactive_limits",
        "exact parameter/function": "lf.Parameters(use_reactive_limits=True)",
        "default setting": "PyPowSyBl docs show use_reactive_limits=True",
        "tested result": "",
        "implication for repository": "Use OpenLoadFlow for operational PF with Q-limit enforcement",
    },
])
display(comparison)
comparison.to_csv(EXP / "loadflow_library_feature_comparison_template.csv", index=False)

# Part D — Recommended first controlled comparison

Use this exact idea before operational controls:

1. Same case data in both libraries.
2. Single connected component.
3. One explicit slack/reference bus.
4. Distributed slack disabled.
5. Reactive limits disabled.
6. Transformer/shunt/phase-shifter controls disabled.
7. Same voltage initialization as far as possible.
8. AC Newton-Raphson explicitly selected.
9. Compare only after confirming data equivalence.

Then run a second operational comparison using PyPowSyBl/OpenLoadFlow with reactive limits and distributed slack enabled.